In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [0]:
spark = SparkSession.builder.getOrCreate()
order_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true") 
    .load("/Volumes/orders/default/orders_csv/orders_100mb.csv")
)

order_df.show(5)
order_df.printSchema()

Q1)  From the dataset, create a result showing `state` and `order_year` with total quantity and total revenue, excluding orders with status `CANCELLED`, and keep only those state–year combinations where total revenue exceeds 550000.

In [0]:
df = (
    order_df
    .filter(F.col("order_status") != "CANCELLED")
    .withColumn("revenue", F.col("quantity")* F.col("price"))
    .groupBy(
        F.col("state"),
        F.year(F.col("order_date")).alias("year")
    )
    .agg(
        F.sum("quantity").alias("total_quantity"),
        F.sum("revenue").alias("total_revenue")
    )
    .filter(F.col("total_revenue") >550000)
)

df.show()

Q2) Add a column `order_value_category` where orders with total amount greater than 400 are labeled `HIGH_VALUE`, between 250 and 400 as `MEDIUM_VALUE`, and the rest as `LOW_VALUE`, then return the count of orders in each category.


In [0]:
df = (
  order_df
  .withColumn(
    "order_value_category",
      F.when(F.col("quantity") >400,"HIGH_VALUE")
      .when((F.col("quantity") >= 250) & (F.col("quantity") <=400),"MEDIUM_VALUE")
      .otherwise("LOW_VALUE")
  )
  .groupBy(F.col('order_value_category'))
  .agg(
      F.count("order_id").alias("order_count")
  )
)
df.show()

In [0]:
df = (
    order_df
    .withColumn("order_amount", F.col("quantity") * F.col("price"))
    .withColumn(
        "order_value_category",
        F.when(F.col("order_amount") > 400, "HIGH_VALUE")
         .when(F.col("order_amount") >= 250, "MEDIUM_VALUE")
         .otherwise("LOW_VALUE")
    )
    .groupBy("order_value_category")
    .agg(
        F.count("*").alias("order_count")
    )
)

df.show()

Q3) For each `state`, calculate the total number of delivered orders and the total number of orders, and compute the delivery percentage for that state.

In [0]:
df = (
  order_df.withColumn(
    "delivered_order_flag",
    F.when(F.col("order_status")== 'DELIVERED',1).otherwise(0)
  )
  .groupby("state")
  .agg(
    F.count("order_id").alias("order_count"),
    F.sum(F.col("delivered_order_flag")).alias("delivered_order_count")
  )
  .withColumn("delivery_percentage",F.round(F.col("delivered_order_count")*100.0/F.col("order_count"),2))
)
df.show()

In [0]:
df = (
    order_df
    .groupBy("state")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(F.when(F.col("order_status") == "DELIVERED", 1).otherwise(0))
         .alias("delivered_orders")
    )
    .withColumn(
        "delivery_percentage",
        F.round((F.col("delivered_orders") * 100.0) / F.col("total_orders"), 2)
    )
)
df.show()